In [3]:
import pandas as pd
import torch
from sklearn.metrics import accuracy_score
from datasets import Dataset
from transformers import (
    BertTokenizer, 
    BertForSequenceClassification, 
    Trainer, 
    TrainingArguments,
    DataCollatorWithPadding
)

# ==========================================
# 1. 核心参数配置
# ==========================================
MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 64
BATCH_SIZE = 8       # CPU 推荐使用 8，防止卡死
EPOCHS = 1           # 建议先跑 1 轮测试全流程，跑通后再改成 3 或 5

# ==========================================
# 2. 数据加载与清洗模块
# ==========================================
def load_tsv_data(file_path):
    """读取 TSV 文件的专用函数"""
    try:
        # 如果你的文件第一行是表头(如 modifier, head, relation)，请把 header=None 改为 header=0
        df = pd.read_csv(
            file_path, 
            sep='\t', 
            header=None,  
            names=['modifier', 'head', 'relation'],
            quoting=3
        )
        df = df.dropna()
        df['modifier'] = df['modifier'].astype(str)
        df['head'] = df['head'].astype(str)
        df['relation'] = df['relation'].astype(str)
        return df
    except Exception as e:
        print(f"❌ 读取文件 {file_path} 失败: {e}")
        return pd.DataFrame() # 返回空表防崩溃

# 分别加载三个数据集
print("正在加载数据集...")
train_df = load_tsv_data("train.tsv")
val_df = load_tsv_data("val.tsv")
test_df = load_tsv_data("test.tsv")

print(f"✅ 数据加载完成 -> 训练集: {len(train_df)}条 | 验证集: {len(val_df)}条 | 测试集: {len(test_df)}条")

# ==========================================
# 3. 标签映射 (严格以 Train 集为准)
# ==========================================
unique_labels = train_df['relation'].unique().tolist()
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}

# 将文本标签转换为数字 ID
train_df['label'] = train_df['relation'].map(label2id)
val_df['label'] = val_df['relation'].map(label2id)
test_df['label'] = test_df['relation'].map(label2id)

# 清理验证集和测试集中可能出现的“训练集没见过的新标签”（防止报错）
val_df = val_df.dropna(subset=['label'])
test_df = test_df.dropna(subset=['label'])

# 确保 label 列是整数类型
train_df['label'] = train_df['label'].astype(int)
val_df['label'] = val_df['label'].astype(int)
test_df['label'] = test_df['label'].astype(int)

# 转换为 Hugging Face 的 Dataset 对象
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

# ==========================================
# 4. 分词器 (Tokenizer) 设置
# ==========================================
print("正在加载分词器...")
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    # 将两个名词拼接：[CLS] modifier [SEP] head [SEP]
    return tokenizer(
        examples["modifier"], 
        examples["head"], 
        truncation=True, 
        max_length=MAX_LENGTH
    )

tokenized_train = train_dataset.map(tokenize_fn, batched=True)
tokenized_val = val_dataset.map(tokenize_fn, batched=True)
tokenized_test = test_dataset.map(tokenize_fn, batched=True)

# ==========================================
# 5. 模型初始化与评估指标
# ==========================================
print("正在加载 BERT 模型结构...")
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=len(unique_labels),
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

# ==========================================
# 6. 训练配置与启动
# ==========================================
training_args = TrainingArguments(
    output_dir="./bert_compound_model",
    eval_strategy="epoch",             # 新版 transformers 使用 eval_strategy
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    load_best_model_at_end=True,       # 训练结束后自动加载 Val Loss 最低的模型权重
    use_cpu=True,                      # 明确告诉框架使用 CPU
    logging_steps=50,                  # 每 50 步打印一次日志
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,        # 新版 transformers 使用 processing_class 替代 tokenizer
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

print("\n" + "="*50)
print("🚀 开始在训练集上微调模型 (Train & Val) ...")
print("="*50)
trainer.train()



正在加载数据集...
✅ 数据加载完成 -> 训练集: 14093条 | 验证集: 940条 | 测试集: 3758条
正在加载分词器...


Map: 100%|██████████| 3758/3758 [00:00<00:00, 66420.27 examples/s]


正在加载 BERT 模型结构...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 978.18it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider tra


🚀 开始在训练集上微调模型 (Train & Val) ...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.800056,0.657119,0.808511


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.51it/s]


TrainOutput(global_step=1762, training_loss=1.1904637794624529, metrics={'train_runtime': 627.0445, 'train_samples_per_second': 22.475, 'train_steps_per_second': 2.81, 'total_flos': 43040507745240.0, 'train_loss': 1.1904637794624529, 'epoch': 1.0})

In [4]:
# ==========================================
# 7. 最终测试与保存模型 (原生 PyTorch 终极拔草版)
# ==========================================
import torch
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding
from sklearn.metrics import accuracy_score

print("\n" + "="*50)
print("🎯 绕过框架 Bug：使用原生 PyTorch 进行最终评估...")
print("="*50)

# 【核心修复】：PyTorch 只懂数学，不懂英文！
# 我们必须把带有英文字母的列删掉，只保留数字特征
cols_to_remove = ["modifier", "head", "relation"]
existing_cols = tokenized_test.column_names
# 把数据集里的这三列拔掉
tokenized_test = tokenized_test.remove_columns([c for c in cols_to_remove if c in existing_cols])

# 将格式转为 PyTorch 需要的标准 tensor
tokenized_test.set_format("torch")

# 准备补齐工具
collate_fn = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

# 创建数据加载器
test_dataloader = DataLoader(tokenized_test, batch_size=BATCH_SIZE, collate_fn=collate_fn)

model.eval()
all_preds = []
all_labels = []

print("🧠 模型正在疯狂做题中...")
with torch.no_grad(): 
    for batch in test_dataloader:
        # 获取输入数据
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        
        # Hugging Face 的 collator 会自动把 'label' 改名为 'labels'
        labels = batch['labels'] if 'labels' in batch else batch['label']
        
        # 组装参数喂给模型
        inputs = {'input_ids': input_ids, 'attention_mask': attention_mask}
        if 'token_type_ids' in batch:
            inputs['token_type_ids'] = batch['token_type_ids']
            
        # 获取模型原始输出
        outputs = model(**inputs)
        
        # 找到概率最大的那个类别的 ID
        preds = torch.argmax(outputs.logits, dim=-1)
        
        # 把这一批的成绩记录下来
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# 自己算分
final_accuracy = accuracy_score(all_labels, all_preds)

print(f"\n📊 【最终成绩单】")
print(f"✅ 测试集准确率 (Test Accuracy): {final_accuracy:.4f}")

# 保存模型 
save_path = "./best_compound_bert"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"\n📁 模型和分词器已成功保存至: {save_path}")
print("🎉 全流程彻底结束！(键盘终于保住了)")


🎯 绕过框架 Bug：使用原生 PyTorch 进行最终评估...
🧠 模型正在疯狂做题中...

📊 【最终成绩单】
✅ 测试集准确率 (Test Accuracy): 0.7853


Writing model shards: 100%|██████████| 1/1 [00:18<00:00, 18.53s/it]



📁 模型和分词器已成功保存至: ./best_compound_bert
🎉 全流程彻底结束！(键盘终于保住了)
